# 02 — Era-weighted calibration

**Question**: the engine fits one stationary profile per asset over its full history. But assets are not stationary across a decade — BTC's vol *halved* as it matured, gold's recent crisis-era vol *exceeds* anything in its history. A full-history profile blends eras that no longer describe the asset. Can fitting on a **recent-era window** predict the *next* year better — and where does era-weighting stop helping?

**Design** (deliberately minimal): `calibrate(..., max_history_years=N)` slices the trailing N years *before* fitting — the era weight is a window choice, every estimator untouched (lab→library: the option graduated to `engine/synthetic/calibration.py`). Era profiles persist as named artifacts (`btcusd_last4y.pkl`), simulatable by stem; the full-history default is untouched.

**Gate** (Phase-1 style, no circularity): candidates are fitted on **train = everything before the last 1 year**, then judged by `profile_gate_check` against **last-1-year targets** they have never seen. Controls: EUR/USD and gold (stationary assets should barely move — if they do, that is informative too).

In [1]:
import numpy as np
import pandas as pd

from synthetic import data as smg_data
from synthetic.calibration import calibrate, log_returns, profile_gate_check, scrub_returns
from synthetic.profile_store import load_profile, save_profile
from synthetic.registry import REGISTRY


def load(key):
    return pd.read_parquet(smg_data.DEFAULT_PROCESSED_DIR / f"{key}_1d.parquet")


def last_year(df, key):
    ppy = REGISTRY[key].ppy_daily
    return df.tail(ppy)


def judge(profile, df, key, n_sims=300, seed=77):
    test = last_year(df, key)
    r_test, _ = scrub_returns(log_returns(test), max_gap_days=REGISTRY[key].scrub_max_gap_days)
    targets = {"mean": r_test.mean(), "vol": r_test.std(), "kurt": r_test.kurt()}
    table = profile_gate_check(profile, targets=targets, n=len(r_test), n_sims=n_sims, seed=seed)
    return targets, table


def vol_row(profile, table):
    r = table[table.stat == "vol"].iloc[0]
    return f"{r['verdict']} (fitted {profile.targets['vol']*100:.2f}%/day vs realized {r['target']*100:.2f}%, band {r['sim 5-95%']})"


# the motivating figure: rolling 2-year annualized vol, per asset
rows = {}
for key in ["eurusd", "xauusd", "btcusd"]:
    df = load(key)
    ppy = REGISTRY[key].ppy_daily
    r = log_returns(df)
    roll = r.rolling(2 * ppy).std() * np.sqrt(ppy)
    rows[key] = {
        "full-sample ann vol": f"{r.std() * np.sqrt(ppy):.1%}",
        "rolling-2y NOW": f"{roll.iloc[-1]:.1%}",
        "rolling-2y min": f"{roll.min():.1%}",
        "rolling-2y max": f"{roll.max():.1%}",
    }
pd.DataFrame(rows).T

,full-sample ann vol,rolling-2y NOW,rolling-2y min,rolling-2y max
eurusd,11.0%,7.0%,5.9%,25.3%
xauusd,18.9%,24.4%,10.1%,31.9%
btcusd,66.6%,44.7%,44.3%,88.9%


The motivation is visible without any fitting: BTC's rolling 2-year vol now sits at its sample **minimum** (its max was double), while gold's sits near its **maximum** — era-dependence runs in both directions. EUR/USD drifts calm too. A single stationary blend cannot be right for all three at once.

In [2]:
# --- BTC: the headline case ---
# candidates are trained on everything BEFORE the last year; judged on the last year
df = load("btcusd")
test = last_year(df, "btcusd")
test_start = pd.Timestamp(test["timestamp"].iloc[0])
train_df = df[df["timestamp"] < test_start].reset_index(drop=True)

full = load_profile(smg_data.DEFAULT_PROCESSED_DIR.parent / "profiles" / "btcusd.pkl")
targets_btc, t_full = judge(full, df, "btcusd")
print(f"BTC last-1y targets: vol={targets_btc['vol']:.4f}  kurt={targets_btc['kurt']:.2f}")
print(f"full-history (2014-26): {vol_row(full, t_full)}")

results = {}
for w in (3.0, 4.0, 5.0):
    prof = calibrate(train_df, asset="btcusd", seed=0, max_history_years=w)
    _, t = judge(prof, df, "btcusd")
    results[w] = (prof, t)
    print(f"era {w}y (train):        {vol_row(prof, t)}")

BTC last-1y targets: vol=0.0238  kurt=6.22
full-history (2014-26): FAIL (fitted 3.44%/day vs realized 2.38%, band [0.0288, 0.0399])


era 3.0y (train):        PASS (fitted 2.48%/day vs realized 2.38%, band [0.0224, 0.0279])


era 4.0y (train):        MARGINAL (fitted 2.78%/day vs realized 2.38%, band [0.0250, 0.0311])


era 5.0y (train):        FAIL (fitted 3.10%/day vs realized 2.38%, band [0.0276, 0.0344])


In [3]:
# --- controls: EUR/USD and gold (stationary assets should barely move) ---
for key in ["eurusd", "xauusd"]:
    df = load(key)
    full = load_profile(smg_data.DEFAULT_PROCESSED_DIR.parent / "profiles" / f"{key}.pkl")
    era = calibrate(df, asset=key, seed=0, max_history_years=5.0)
    targets, t_full = judge(full, df, key)
    _, t_era = judge(era, df, key)
    print(f"{key}: last-1y vol={targets['vol']:.4f}")
    print(f"  full-history: {vol_row(full, t_full)}")
    print(f"  era 5y:       {vol_row(era, t_era)}")

/home/sebastian-salles/Projects/synthetic-market-generator/.venv/lib/python3.12/site-packages/arch/univariate/volatility.py:1233: RuntimeWarning: invalid value encountered in scalar divide
  scale = np.mean(resids**2) / (target ** (2.0 / power))
/home/sebastian-salles/Projects/synthetic-market-generator/.venv/lib/python3.12/site-packages/arch/univariate/distribution.py:553: RuntimeWarning: divide by zero encountered in log
  lls -= 0.5 * (log(sigma2))
/home/sebastian-salles/Projects/synthetic-market-generator/.venv/lib/python3.12/site-packages/arch/univariate/distribution.py:554: RuntimeWarning: invalid value encountered in divide
  lls -= ((nu + 1) / 2) * (log(1 + (resids**2.0) / (sigma2 * (nu - 2))))
/home/sebastian-salles/Projects/synthetic-market-generator/.venv/lib/python3.12/site-packages/arch/univariate/distribution.py:554: RuntimeWarning: divide by zero encountered in divide
  lls -= ((nu + 1) / 2) * (log(1 + (resids**2.0) / (sigma2 * (nu - 2))))
/home/sebastian-salles/Projects

eurusd: last-1y vol=0.0035
  full-history: FAIL (fitted 0.57%/day vs realized 0.35%, band [0.0048, 0.0077])
  era 5y:       FAIL (fitted 0.47%/day vs realized 0.35%, band [0.0048, 0.0086])


xauusd: last-1y vol=0.0184
  full-history: FAIL (fitted 1.14%/day vs realized 1.84%, band [0.0088, 0.0146])
  era 5y:       FAIL (fitted 1.20%/day vs realized 1.84%, band [0.0093, 0.0166])


In [4]:
# --- ship the era artifacts (fitted on the FULL window incl. the last year) ---
saved = {}
for w in (3.0, 4.0):
    prof = calibrate(load("btcusd"), asset="btcusd", seed=0, max_history_years=w)
    # explicit suffixed path: profile.asset stays 'btcusd', so the default
    # save path would collide with the full-history artifact
    saved[w] = save_profile(prof, smg_data.DEFAULT_PROCESSED_DIR.parent / 'profiles' / f'btcusd_last{w:g}y.pkl')
    print(f"era {w}y fitted vol {prof.targets['vol']:.4f} -> {saved[w]}")
print("\nprofiles dir now serves:", sorted(p.stem for p in (smg_data.DEFAULT_PROCESSED_DIR.parent / 'profiles').glob('*.pkl')))

era 3.0y fitted vol 0.0247 -> /home/sebastian-salles/Projects/synthetic-market-generator/data/profiles/btcusd_last3y.pkl


era 4.0y fitted vol 0.0245 -> /home/sebastian-salles/Projects/synthetic-market-generator/data/profiles/btcusd_last4y.pkl

profiles dir now serves: ['btcusd', 'btcusd_last3y', 'btcusd_last4y', 'eurusd', 'xauusd']


## Verdict

**1. Era-dependence is real, bidirectional, and large.** Judged on the unseen last year, BTC's full-history profile **fails its own gate** on vol (fitted 3.44%/day vs a realized 2.38%, outside the [2.88, 3.99] band) — the same class of FAIL the holdout study produced in Phase 1, now driven by genuine non-stationarity rather than data errors. Gold fails in the *opposite* direction: its crisis-era year (1.84%/day) exceeds every window of its history.

**2. Era-weighting closes the gate for BTC — and the window length is the dial.** 3-year window: **PASS** (fitted 2.48%, band [2.24, 2.79]). 4-year: MARGINAL (2.78%, band [2.50, 3.11]). 5-year: FAIL (3.10%) — longer windows re-admit the high-vol past. The tradeoff is the classic one: shorter windows track the present at the cost of noisier weakly-identified parameters (drift, dwell — already flagged as uncertain in Phase 1). Shipped: `btcusd_last3y` and `btcusd_last4y` as named artifacts alongside the untouched full-history default.

**3. Controls behave, and the limit is named.** EUR/USD improves with era-weighting (0.57→0.47%/day fitted; the last year was calm even by recent standards — its MARGINAL band edge is honest), so era artifacts are available for it too. But **no backward-looking window captures a genuinely unprecedented regime** (gold's 2026 year) — that is the irreducible limit of empirical calibration, and it is exactly what Phase 5's scenario injection is for: *stress comes from scenarios, not from stretching history.*

**Recommendation**: default stays full-history for stationary assets; era-weighted profiles (`_last{N}y`) are first-class, simulatable artifacts for maturing assets — chosen per asset, documented per profile (`history_years`), and gated per window — not a global switch.